In [2]:
import numpy as pd
import pandas as pd
import plotly.offline as pyo
import plotly.graph_objs as go

In [3]:
match = pd.read_csv('D:\dataset\matches.csv')
delivery = pd.read_csv('D:\dataset\deliveries.csv')
ipl = delivery.merge(match,left_on='match_id',right_on='id')
ipl.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen


In [4]:
#scattter plot are drawn between to continous variables
#problem :- we are going to fraw a sctterplot between batman avg(x axis) and
#batman strike rate(y axis)of the top 50 batman in IPL (all time)
ipl.columns

Index(['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball',
       'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs',
       'total_runs', 'extras_type', 'is_wicket', 'player_dismissed',
       'dismissal_kind', 'fielder', 'id', 'season', 'city', 'date',
       'match_type', 'player_of_match', 'venue', 'team1', 'team2',
       'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin',
       'target_runs', 'target_overs', 'super_over', 'method', 'umpire1',
       'umpire2'],
      dtype='object')

In [5]:
# AVG vs SR GRAPH OF TOP BATSMAN (in terms of total runs)

# Fetching a new dataframe with top 50 batsman
top50 = ipl.groupby('batter')['batsman_runs'].sum() \
          .sort_values(ascending=False) \
          .head(50).index.tolist()

new_ipl = ipl[ipl['batter'].isin(top50)]

In [6]:
# Calculating Strike Rate

runs = new_ipl.groupby('batter')['batsman_runs'].sum()
balls = new_ipl.groupby('batter')['batsman_runs'].count()

sr = (runs / balls) * 100
sr = sr.reset_index()

sr

,batter,batsman_runs
0,AB de Villiers,148.580442
1,AD Russell,164.224422
2,AJ Finch,123.349057
3,AM Rahane,120.321410
4,AT Rayudu,124.584527
5,BB McCullum,126.848592
6,CH Gayle,142.121729
7,DA Miller,134.684477
8,DA Warner,135.429986
9,DR Smith,132.279534


In [7]:
#calculating avg
#avg = (total number of runs )/(nu,mber of outs)
#calculateing number of outs for top 50 batsman
out = ipl[ipl['player_dismissed'].isin(top50)]

nouts = out['player_dismissed'].value_counts()

avg = runs/nouts

avg= avg.reset_index()
avg.rename(columns={'index':'batter',0:'avg'},inplace=True)
avg = avg.merge(sr,on='batter')
avg

,batter,avg,batsman_runs
0,AB de Villiers,39.853846,148.580442
1,AD Russell,28.930233,164.224422
2,AJ Finch,24.904762,123.349057
3,AM Rahane,30.142857,120.321410
4,AT Rayudu,28.051613,124.584527
5,BB McCullum,27.711538,126.848592
6,CH Gayle,39.658730,142.121729
7,DA Miller,35.658537,134.684477
8,DA Warner,40.042683,135.429986
9,DR Smith,28.392857,132.279534


In [8]:
#plot sctter plot here
trace = go.Scatter(x=avg['avg'],y=avg['batsman_runs'],mode='markers',text=avg['batter'],marker={'color':'#00a65a','size':16})

data=[trace]

layout=go.Layout(title='Batman Avg vs SR',xaxis={'title':'Batman average'},yaxis={'title':'batman strike rate'})

fig=go.Figure(data=data,layout=layout)

pyo.plot(fig)

'temp-plot.html'

In [9]:
#year by year batman performance

single = ipl[ipl['batter']=='V Kohli']
performance = single.groupby('season')['batsman_runs'].sum().reset_index()
performance

single1 = ipl[ipl['batter']=='MS Dhoni']
performance1 = single1.groupby('season')['batsman_runs'].sum().reset_index()
performance1

,season,batsman_runs
0,2007/08,414
1,2009,332
2,2009/10,287
3,2011,392
4,2012,358
5,2013,461
6,2014,371
7,2015,372
8,2016,284
9,2017,290


In [10]:
#plot line chart here
trace = go.Scatter(x=performance['season'],
                   y=performance['batsman_runs'],
                   mode='lines+markers',marker={'color':'#00a65a'},name='virat kohli')


trace1 = go.Scatter(x=performance1['season'],
                   y=performance1['batsman_runs'],
                   mode='lines+markers',name='ms dhoni')

data = [trace,trace1]

layout = go.Layout(title='Year by Year performance'
                   ,xaxis={'title':'Season'},
                   yaxis={'title':'total runs'})

fig = go.Figure(data=data,layout=layout)

pyo.plot(fig)

'temp-plot.html'

In [11]:
# miltiple line charts

def batman_comp(*name):
    data=[]
    for i in name:
        single=ipl[ipl['batter']==i]
        performance=single.groupby('season')['batsman_runs'].sum().reset_index()

        trace=go.Scatter(x=performance['season'],y=performance['batsman_runs'],
                         mode='lines+ markers',name=i)

        data.append(trace)

        layout=go.Layout(title='Batman record comparator',
                        xaxis={'title':'season'},
                        yaxis={'title':'Runs'})
        fig=go.Figure(data=data,layout=layout)

        pyo.plot(fig,filename='year_by_year')

In [12]:
batman_comp('V Kohli','RG Sharma','DA Warner')

C:\Users\upma dubey\anaconda3\envs\booksenv\lib\site-packages\plotly\offline\offline.py:556: UserWarning: Your filename `year_by_year` didn't end with .html. Adding .html to the end of your file.
  warnings.warn(


C:\Users\upma dubey\anaconda3\envs\booksenv\lib\site-packages\plotly\offline\offline.py:556: UserWarning: Your filename `year_by_year` didn't end with .html. Adding .html to the end of your file.
  warnings.warn(


C:\Users\upma dubey\anaconda3\envs\booksenv\lib\site-packages\plotly\offline\offline.py:556: UserWarning: Your filename `year_by_year` didn't end with .html. Adding .html to the end of your file.
  warnings.warn(


In [13]:
#Bar plot 
#one categorical and 1 numerical values
top10 = ipl.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(10).index.tolist()
top10_df=ipl[ipl['batter'].isin(top10)]

In [14]:
top10_score = top10_df.groupby('batter')['batsman_runs'].sum().reset_index()
top10_score

,batter,batsman_runs
0,AB de Villiers,5181
1,CH Gayle,4997
2,DA Warner,6567
3,KD Karthik,4843
4,MS Dhoni,5243
5,RG Sharma,6630
6,RV Uthappa,4954
7,S Dhawan,6769
8,SK Raina,5536
9,V Kohli,8014


In [18]:
trace=go.Bar(x=top10_score['batter'],y=top10_score['batsman_runs'])
data = [trace]
layout=go.Layout(title='top 10 ipl batter',xaxis={'title':'batters name'},yaxis={'title':'total runs'})
fig=go.Figure(data=data,layout=layout)
pyo.plot(fig)

'temp-plot.html'

In [16]:
match_agg = delivery.groupby(['match_id'])['total_runs'].sum().reset_index()
season_wise=match_agg.merge(match,left_on='match_id',right_on='id')[['match_id','total_runs','season']]
season_wise

,match_id,total_runs,season
0,335982,304,2007/08
1,335983,447,2007/08
2,335984,261,2007/08
3,335985,331,2007/08
4,335986,222,2007/08
...,...,...,...
1090,1426307,429,2024
1091,1426309,323,2024
1092,1426310,346,2024
1093,1426311,314,2024


In [28]:
trace = go.Box(
    x=season_wise[season_wise['season'] == '2017']['total_runs'],
    name='2017',
    marker={'color': '#00a65a'}
)

trace1 = go.Box(
    x=season_wise[season_wise['season'] == '2007/08']['total_runs'],
    name='2007/08',
    marker={'color': '#ff5733'}
)

data = [trace, trace1]

layout = go.Layout(
    title='Total Score Analysis',
    xaxis={'title': 'Total Score'}
)

fig = go.Figure(data=data, layout=layout)

pyo.plot(fig)

'temp-plot.html'

In [30]:
import plotly.figure_factory as ff

hist_data = [avg['avg'],avg['batsman_runs']]

group_labels=['Average','Strike Rate']

fig=ff.create_distplot(hist_data,group_labels,bin_size=[10,20])

pyo.plot(fig)

'temp-plot.html'

In [41]:
x = delivery.groupby('batter')['batsman_runs'].count()>150
x = x[x].index.tolist()

new=delivery[delivery['batter'].isin(x)]

runs = new.groupby('batter')['batsman_runs'].sum()
balls=new.groupby('batter')['batsman_runs'].count()

sr=(runs/balls)*100

sr=sr.reset_index()
sr


,batter,batsman_runs
0,A Ashish Reddy,142.857143
1,A Badoni,125.544554
2,A Manohar,127.624309
3,A Mishra,86.590909
4,A Symonds,124.711908
...,...,...
235,Y Venugopal Rao,113.872832
236,YBK Jaiswal,146.757991
237,YK Pathan,138.046272
238,YV Takawale,104.918033


In [47]:
trace=go.Histogram(x=sr['batsman_runs'],xbins={'size':2,'start':50,'end':100})
data=[trace]
layout=go.Layout(title='Strike Rate Analysis',
                xaxis={'title':'Strike RAtes'})
fig=go.Figure(data=data,layout=layout)
pyo.plot(fig)

'temp-plot.html'

In [49]:
six = delivery[delivery['batsman_runs']==6]
six=six.groupby(['batting_team','over'])['batsman_runs'].count().reset_index()

six

,batting_team,over,batsman_runs
0,Chennai Super Kings,0,9
1,Chennai Super Kings,1,36
2,Chennai Super Kings,2,67
3,Chennai Super Kings,3,71
4,Chennai Super Kings,4,75
...,...,...,...
371,Sunrisers Hyderabad,15,56
372,Sunrisers Hyderabad,16,59
373,Sunrisers Hyderabad,17,73
374,Sunrisers Hyderabad,18,94


In [54]:
import plotly.offline as pyo

trace = go.Heatmap(
    x=six['batting_team'],
    y=six['over'],
    z=six['batsman_runs']
)

data = [trace]

layout = go.Layout(
    title='Six Heatmap'
)

fig = go.Figure(
    data=data,
    layout=layout
)

pyo.plot(fig)

'temp-plot.html'